# Co-occurrence Networks

**Co-occurrence networks** are representations of the aggregated interconnection of items, based on their paired presence in a specified context.

Items are represented by nodes, and edges exist between pairs of items if they appear together in the same context at least once. These networks are particularly useful for identifying patterns of association and relationships that might not be immediately apparent in raw data.

In [ ]:
from collections import Counter
import itertools, json
import networkx as nx
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
%matplotlib inline

## Simple Example

Firstly, we look at a very simple example of a generic co-occurrence network, where we have 7 items (A–G) that appear in 6 different contexts.

In [ ]:
contexts = {}
contexts[1] = ["A", "B", "C"]
contexts[2] = ["C", "D"]
contexts[3] = ["C", "E"]
contexts[4] = ["C", "D", "E", "F"]
contexts[5] = ["A", "B"]
contexts[6] = ["E", "F", "G"]

To create the network, we process each context to find the unique pairs in each context using the `itertools.combinations` function:

In [ ]:
pair_counts = Counter()
for c in contexts:
    # use itertools to find all unique combinations
    context_pairs = list(itertools.combinations(contexts[c], r=2))
    print(context_pairs)
    # update the pair counts
    for pair in context_pairs:
        pair_counts[pair] += 1

We can look at the individual pair counts - i.e. the number of times each pair of items occurred in the same context. These counts form the basis for the edge weights in our network.

In [ ]:
# display the counts
for pair in pair_counts:
    print(f"{pair} = {pair_counts[pair]}")

Now create the actual co-occurrence network from the pair counts using NetworkX:

In [ ]:
g = nx.Graph()
# add a new edge for each pair, where the weight is the count for that pair
for pair in pair_counts:
    g.add_edge(pair[0], pair[1], weight=pair_counts[pair])

# check the size of the new network
print(f"Network has {g.number_of_nodes()} nodes and {g.number_of_edges()} edges")

Visualise the new network, with edge weights shown as labels on the edges. This representation clearly shows the strength of relationships between different items:

In [ ]:
plt.figure(figsize=(6, 5))
plt.margins(0.1, 0.1)
# apply layout algorithm to calculate node positions
pos = nx.spring_layout(g)
# draw the nodes and edges
nx.draw(g, pos, 
        with_labels=True, 
        node_size=1300, 
        font_size=13, 
        node_color="#abeeaa")
# now add the edge weights as labels
labels = nx.get_edge_attributes(g, "weight")
nx.draw_networkx_edge_labels(g, pos,edge_labels=labels)
plt.show()

## Co-purchasing Network

Next, we will look at a specific type of co-occurrence network, called a **co-purchasing network**. Here nodes represent products, and an edge exists between two products if they were both bought by the same customer. This type of network is useful for understanding consumer behaviour and product associations.

First read the JSON file containing a sample of customer purchasing records:

In [ ]:
with open("purchases.json", "r") as fin:
    data = json.load(fin)
    print(f"Read data for {len(data)} customers")

Process each customer (context) to find the unique pairs of products:

In [ ]:
pair_counts = Counter()
for customer in data:
    # use itertools to find all unique combinations
    customer_pairs = list(itertools.combinations(customer["purchases"], r=2))
    # update the pair counts
    for pair in customer_pairs:
        pair_counts[pair] += 1
print(pair_counts)

Create the network from the pair frequencies. We add an edge for each unique pair of products, where the edge weight is the frequency. Note that the nodes will be automatically added to the network when we add the edges using NetworkX.

In [ ]:
g = nx.Graph()
# create an edge from each pair
for pair in pair_counts:
    g.add_edge(pair[0], pair[1], weight=pair_counts[pair])

# check the size of the new network
print(f"Network has {g.number_of_nodes()} nodes and {g.number_of_edges()} edges")

How dense is the network and how many components are there?

In [ ]:
print(f"Density={nx.density(g):.2f} Components={nx.number_connected_components(g)}")

Visualise the new network:

In [ ]:
plt.figure(figsize=(13,9)) 
plt.margins(0.1, 0.1)
nx.draw(g, 
        with_labels=True, 
        node_size=1000, 
        font_size=11, 
        node_color="#abeeaa")
plt.show()

We could extract the ego network of a given product, and use this as a source of similar product recommendations. An ego network shows the immediate neighbourhood of a focal node:

In [ ]:
product = "Fire TV Cube"
eg = nx.ego_graph(g, product)

Visualise the new ego network:

In [ ]:
plt.figure(figsize=(8,7)) 
plt.margins(0.1, 0.1)
# apply layout algorithm to calculate node positions
pos = nx.spring_layout( eg )
# draw the nodes and edges
nx.draw(eg, pos, 
        with_labels=True, 
        node_size=1400, 
        font_size=13, 
        node_color="#abeeaa")
# draw the ego in red, with larger node size
nx.draw_networkx_nodes(eg, pos, 
                       nodelist=[product], 
                       node_size=2500, 
                       node_color="#FA8072")
# now add the edge weights as labels
labels = nx.get_edge_attributes(eg, "weight")
nx.draw_networkx_edge_labels(eg, pos,edge_labels=labels)
plt.show()

The neighbours of the ego product, which have the highest edge weights, could potentially be recommended as relevant products to a customer. This approach forms the basis of many recommendation systems.

## Collaboration Network

As our final example, we will look at another type of co-occurrence network, called a **collaboration network**. In this type, nodes typically represent individuals, and an edge exists between two individuals if they have collaborated together at least once in some context (e.g. jointly worked on the same project, jointly contributed to the same code repository, jointly co-authored the same research paper).

In this specific case, we will look at collaboration activity between a group of university students across three different group assignments.

First, use Pandas to read in the dataset in comma-separated format:

In [ ]:
df = pd.read_csv("groups.csv")
df

Here each context is a group (A or B or C) for a given assignment (1 or 2 or 3), as stored in the 'members' column of the DataFrame. We will count the pairs of collaborations for each value for that column.

In [ ]:
pair_counts = Counter()
for i, row in df.iterrows():
    # note that we need to split the strings in the members column with the appropriate separator
    members = row["members"].split(" + ")
    # use itertools to find all unique combinations for the members of each assignment group
    context_pairs = list(itertools.combinations(members, r=2))
    # update the pair counts
    for pair in context_pairs:
        pair_counts[pair] += 1

Now we create the network from the collaboration pair frequencies. We add an edge for each unique pair of students, where the edge weight is the frequency of collaboration (i.e. how many times 2 students were in the same group).

In [ ]:
# add the edges - the nodes get added automatically
g = nx.Graph()
for pair in pair_counts:
    g.add_edge(pair[0], pair[1], weight=pair_counts[pair])
    
# check the size of the network
print(f"Network has {g.number_of_nodes()} nodes and {g.number_of_edges()} edges")

We can visualise the new assignment collaboration network, with weights as labels on the edges:

In [ ]:
plt.figure(figsize=(6, 5.5))
# apply layout algorithm to calculate node positions
pos = nx.spring_layout(g)
# draw the nodes and edges
nx.draw(g, pos, 
        with_labels=True, 
        node_size=1200, 
        font_size=12, 
        node_color="#bebbeb")
# now add the edge weights as labels
labels = nx.get_edge_attributes(g, "weight")
nx.draw_networkx_edge_labels(g, pos,edge_labels=labels)
plt.show()